# 14. In-Field Protein Prediction Validation

Validate the best GPC prediction model (Random Forest, `mo_p1` temporal window) on
**186 independent in-field protein samples** from 16 wheat fields.

This tests whether the field-level model can predict **within-field** protein variability.

**Pipeline:**
1. Load GEE per-image S2 + daily ERA5 exports for in-field samples
2. Peak detection (GCVI double logistic) per field
3. Temporal alignment to peak-relative days
4. Aggregate to `mo_p1` window (days +16 to +45) with mean/std/p10 stats
5. Combine with elevation + soil features
6. Train RF on all field-level training data, predict on in-field samples
7. Evaluate R², RMSE, per-field analysis

In [ ]:
import sys
import glob
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

# Add project root to path
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config, get_feature_type
from src.data_loading import (
    load_spectral, load_meteo, load_elevation,
    load_static_features, merge_all_static
)
from src.peak_detection import detect_peaks
from src.temporal_alignment import (
    normalize_to_peak, aggregate_periods, compute_derived_meteo,
    generate_monthly_windows, _compute_agg
)

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 6)

print('Project root:', PROJECT_ROOT)

## Configuration

In [ ]:
# === Paths ================================================================

# Model
MODEL_PATH = PROJECT_ROOT / 'models/temporal_combinations/selected_monthly_analysis/models/mo_p1__to__mo_p1_RandomForest.joblib'

# In-field GEE exports (download from Google Drive after running GEE script)
INFIELD_S2_DIR = PROJECT_ROOT / 'data/raw/spectral/GEE_Infield_Samples'
INFIELD_METEO_DIR = PROJECT_ROOT / 'data/raw/meteo/GEE_Infield_Samples'
INFIELD_ELEV_PATH = PROJECT_ROOT / 'data/raw/drive_download/infield_elevation_stats.csv'

# In-field metadata and soil
GPC_PREDICTION_ROOT = Path(r'C:/Users/Vlasis/Repos/GPC_Prediction')
INFIELD_METADATA_PATH = GPC_PREDICTION_ROOT / 'data/processed/infield_samples_for_modeling.csv'
INFIELD_SOIL_PATH = GPC_PREDICTION_ROOT / 'data/processed/two_layers_infield_samples_soil_features_saxton_by_field.csv'

# === Load config ==========================================================
config = load_config()
config['aggregation']['mode'] = 'monthly'  # Force monthly mode for mo_p1

print('In-field S2 dir:', INFIELD_S2_DIR)
print('In-field meteo dir:', INFIELD_METEO_DIR)
print('Model path:', MODEL_PATH)

## 1. Load Model Metadata

In [ ]:
model_data = joblib.load(MODEL_PATH)

selected_features = model_data['features']
best_params = model_data['best_params']

print(f"Model: {model_data['combination']} - RandomForest")
print(f"CV R2: {model_data['cv_r2']:.4f}")
print(f"CV RMSE: {model_data['cv_rmse']:.4f}")
print(f"\nSelected features ({len(selected_features)}):")
for i, f in enumerate(selected_features, 1):
    print(f"  {i:2d}. {f}")
print(f"\nBest params: {best_params}")

## 2. Build Training Feature Matrix

Reconstruct the full training dataset using the pipeline modules.

In [ ]:
# Load training data from pipeline
print('Loading training spectral data...')
train_spectral = load_spectral(config['data']['spectral_dir'])

print('Loading training meteo data...')
train_meteo = load_meteo(config['data']['meteo_dir'])

# Peak detection
print('Running peak detection...')
train_peak_df, train_params_df = detect_peaks(train_spectral, config)

# Temporal alignment
print('Normalizing to peak...')
train_spectral_norm = normalize_to_peak(train_spectral, train_peak_df)
train_meteo_norm = normalize_to_peak(train_meteo, train_peak_df)
train_meteo_norm = compute_derived_meteo(train_meteo_norm)

# Aggregate periods
print('Aggregating periods...')
train_aggregated = aggregate_periods(train_spectral_norm, train_meteo_norm, config)

# Load static features
print('Loading static features...')
static_df, protein_df = load_static_features(
    config['data']['static_features_path'],
    config['soil']['columns'],
    config['soil'].get('static_columns')
)
elevation_df = load_elevation(config['data']['field_elevation_path'])
merged_static = merge_all_static(static_df, protein_df, elevation_df)

# Combine
train_df = train_aggregated.merge(merged_static, on='field_key')

# Check all features present
missing = [f for f in selected_features if f not in train_df.columns]
if missing:
    print(f'WARNING: Missing training features: {missing}')
else:
    print(f'All {len(selected_features)} features present in training data')

print(f'\nTraining data: {len(train_df)} fields x {len(train_df.columns)} columns')
train_df[selected_features + ['protein_pct']].describe().round(3)

## 3. Load In-Field Sample Metadata

In [ ]:
# Load sample metadata with protein values
infield_meta = pd.read_csv(INFIELD_METADATA_PATH)
infield_meta['field_key'] = infield_meta['field_key'].astype(str)
infield_meta['sample_key'] = infield_meta['sample_key'].astype(str)

print(f'In-field samples: {len(infield_meta)}')
print(f'Fields: {infield_meta["field_key"].nunique()}')
print(f'Protein range: {infield_meta["protein_pct"].min():.1f} - {infield_meta["protein_pct"].max():.1f}%')
print(f'\nSamples per field:')
print(infield_meta.groupby('field_key').size().to_string())

## 4. Load In-Field GEE Exports

**Prerequisite**: Run `src/gee_scripts/infield_samples_extraction.js` in GEE Code Editor
and download the exported CSVs from Google Drive.

In [ ]:
# Load in-field S2 data (per-image observations)
s2_files = sorted(glob.glob(str(INFIELD_S2_DIR / 'infield_daily_s2_*.csv')))
print(f'Found {len(s2_files)} S2 files')

if not s2_files:
    raise FileNotFoundError(
        f'No S2 files found in {INFIELD_S2_DIR}. '
        'Download GEE exports from Google Drive first.'
    )

infield_s2 = pd.concat([pd.read_csv(f) for f in s2_files], ignore_index=True)
infield_s2['date'] = pd.to_datetime(infield_s2['date'])
infield_s2['field_key'] = infield_s2['field_key'].astype(str)
infield_s2['sample_key'] = infield_s2['sample_key'].astype(str)
infield_s2.sort_values(['sample_key', 'date'], inplace=True)
infield_s2.reset_index(drop=True, inplace=True)

# Drop rows where all spectral values are NaN (fully cloudy)
spectral_cols = [c for c in infield_s2.columns if c not in ['sample_key', 'field_key', 'date']]
before = len(infield_s2)
infield_s2 = infield_s2.dropna(subset=spectral_cols, how='all')
print(f'Dropped {before - len(infield_s2)} fully-cloudy rows')

print(f'\nIn-field S2 data: {len(infield_s2)} observations')
print(f'  Samples: {infield_s2["sample_key"].nunique()}')
print(f'  Fields: {infield_s2["field_key"].nunique()}')
print(f'  Date range: {infield_s2["date"].min().date()} to {infield_s2["date"].max().date()}')
print(f'  Unique dates: {infield_s2["date"].nunique()}')

In [ ]:
# Load in-field meteo data (daily)
meteo_files = sorted(glob.glob(str(INFIELD_METEO_DIR / 'infield_daily_meteo_*.csv')))
print(f'Found {len(meteo_files)} meteo files')

infield_meteo = pd.concat([pd.read_csv(f) for f in meteo_files], ignore_index=True)
infield_meteo['date'] = pd.to_datetime(infield_meteo['date'])
infield_meteo['field_key'] = infield_meteo['field_key'].astype(str)
infield_meteo['sample_key'] = infield_meteo['sample_key'].astype(str)
infield_meteo.sort_values(['sample_key', 'date'], inplace=True)
infield_meteo.reset_index(drop=True, inplace=True)

print(f'\nIn-field meteo data: {len(infield_meteo)} rows')
print(f'  Date range: {infield_meteo["date"].min().date()} to {infield_meteo["date"].max().date()}')
print(f'  Columns: {[c for c in infield_meteo.columns if c not in ["sample_key", "field_key", "date"]]}')

In [ ]:
# Load in-field elevation stats
infield_elev = pd.read_csv(INFIELD_ELEV_PATH)
infield_elev['field_key'] = infield_elev['field_key'].astype(str)
infield_elev['sample_key'] = infield_elev['sample_key'].astype(str)

print(f'Elevation data: {len(infield_elev)} samples')
print(f'Columns: {list(infield_elev.columns)}')
infield_elev.head()

In [ ]:
# Load in-field soil data
# The soil CSV has a numeric sample_id (0-185) that does NOT match sample_key.
# Match to metadata via rounded lon/lat coordinates instead.
infield_soil = pd.read_csv(INFIELD_SOIL_PATH)
infield_soil['field_key'] = infield_soil['field_key'].astype(str)

# Required soil features
soil_features = [
    'soil_clay_ratio', 'soil_sub_ksat_log10', 'soil_subsoil_clay',
    'soil_topsoil_cec', 'soil_topsoil_om'
]

available_soil = [c for c in soil_features if c in infield_soil.columns]
missing_soil = [c for c in soil_features if c not in infield_soil.columns]
if missing_soil:
    print(f'WARNING: Missing soil features: {missing_soil}')

# Join soil to metadata via rounded coordinates (5 decimal places ~ 1m precision)
infield_soil['lon_r'] = infield_soil['lon'].round(5)
infield_soil['lat_r'] = infield_soil['lat'].round(5)
meta_for_join = infield_meta[['sample_key', 'field_key', 'lon', 'lat']].copy()
meta_for_join['lon_r'] = meta_for_join['lon'].round(5)
meta_for_join['lat_r'] = meta_for_join['lat'].round(5)

infield_soil_clean = meta_for_join[['sample_key', 'lon_r', 'lat_r']].merge(
    infield_soil[['lon_r', 'lat_r'] + available_soil],
    on=['lon_r', 'lat_r'], how='left'
).drop(columns=['lon_r', 'lat_r'])

n_matched = infield_soil_clean[available_soil[0]].notna().sum()
print(f'Soil data: {n_matched}/{len(infield_soil_clean)} samples matched via coordinates')
print(f'Available: {available_soil}')
infield_soil_clean.head()

## 5. Peak Detection

Detect GCVI peak per field using double logistic fit.
Aggregate sample-level S2 data to field level for peak detection.

In [ ]:
# Aggregate S2 data to field level for peak detection
# (average across all samples within each field for each date)
field_s2 = infield_s2.groupby(['field_key', 'date']).mean(numeric_only=True).reset_index()

print(f'Field-level S2 data: {len(field_s2)} rows, {field_s2["field_key"].nunique()} fields')

# Run peak detection
infield_peak_df, infield_params_df = detect_peaks(field_s2, config)

print(f'\nPeaks detected for {len(infield_peak_df)} fields:')
for _, row in infield_peak_df.iterrows():
    flag = ' (ANOMALOUS)' if row.get('peak_anomalous', False) else ''
    print(f"  Field {row['field_key']}: peak = {row['peak_date']}{flag}")

# Plot GCVI profiles with peaks
n_fields = field_s2['field_key'].nunique()
ncols = 4
nrows = (n_fields + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3 * nrows), sharex=True, sharey=True)
axes = axes.flatten()

for idx, (fk, grp) in enumerate(field_s2.groupby('field_key')):
    if idx >= len(axes):
        break
    ax = axes[idx]
    ax.scatter(grp['date'], grp['GCVI'], s=10, alpha=0.5)
    peak_row = infield_peak_df[infield_peak_df['field_key'] == fk]
    if len(peak_row) > 0:
        peak_date = pd.Timestamp(peak_row.iloc[0]['peak_date'])
        ax.axvline(peak_date, color='red', linestyle='--', alpha=0.7, label='Peak')
    ax.set_title(f'Field {fk}', fontsize=9)
    ax.tick_params(labelsize=7)

# Hide unused axes
for idx in range(n_fields, len(axes)):
    axes[idx].set_visible(False)

fig.suptitle('GCVI Profiles with Detected Peaks (In-Field Data)', fontsize=14)
fig.autofmt_xdate()
plt.tight_layout()
plt.savefig('../figures/14_infield_gcvi_peaks.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Temporal Alignment & Aggregation to mo_p1

Normalize dates to peak-relative days, then aggregate **per sample** within
the `mo_p1` window (days +16 to +45 relative to peak).

In [ ]:
# Normalize S2 and meteo dates to peak-relative days
# Note: normalize_to_peak maps by field_key
infield_s2_norm = normalize_to_peak(infield_s2, infield_peak_df)
infield_meteo_norm = normalize_to_peak(infield_meteo, infield_peak_df)

print(f'Normalized S2: {len(infield_s2_norm)} rows')
print(f'Normalized meteo: {len(infield_meteo_norm)} rows')
print(f'Normalized day range (S2): '
      f'[{infield_s2_norm["normalized_day"].min()}, '
      f'{infield_s2_norm["normalized_day"].max()}]')

In [ ]:
# Get mo_p1 window boundaries
mo_windows = generate_monthly_windows(config)
mo_p1_window = [w for w in mo_windows if w[0] == 'mo_p1'][0]
p1_name, p1_start, p1_end = mo_p1_window
print(f'mo_p1 window: days [{p1_start}, {p1_end}]')

# Filter to mo_p1 window
s2_p1 = infield_s2_norm[
    (infield_s2_norm['normalized_day'] >= p1_start) &
    (infield_s2_norm['normalized_day'] <= p1_end)
].copy()

meteo_p1 = infield_meteo_norm[
    (infield_meteo_norm['normalized_day'] >= p1_start) &
    (infield_meteo_norm['normalized_day'] <= p1_end)
].copy()

print(f'\nS2 observations in mo_p1: {len(s2_p1)}')
print(f'  Per sample: min={s2_p1.groupby("sample_key").size().min()}, '
      f'max={s2_p1.groupby("sample_key").size().max()}, '
      f'mean={s2_p1.groupby("sample_key").size().mean():.1f}')
print(f'\nMeteo observations in mo_p1: {len(meteo_p1)}')
print(f'  Per sample: min={meteo_p1.groupby("sample_key").size().min()}, '
      f'max={meteo_p1.groupby("sample_key").size().max()}, '
      f'mean={meteo_p1.groupby("sample_key").size().mean():.1f}')

In [ ]:
# Aggregate spectral + meteo features PER SAMPLE within mo_p1
# (Custom aggregation because aggregate_periods() groups by field_key)
agg_cfg = config['aggregation']['agg_functions']
exclude_cols = {'sample_key', 'field_key', 'date', 'normalized_day'}
s2_feature_cols = [c for c in s2_p1.columns if c not in exclude_cols]
meteo_feature_cols = [c for c in meteo_p1.columns if c not in exclude_cols]

all_sample_records = []

for sk in infield_meta['sample_key'].unique():
    record = {'sample_key': sk}

    # S2 spectral aggregation
    s2_sample = s2_p1[s2_p1['sample_key'] == sk]
    for feat in s2_feature_cols:
        if feat not in s2_sample.columns:
            continue
        feat_type = get_feature_type(feat)
        stats = agg_cfg.get(feat_type, ['mean'])
        values = s2_sample[feat].dropna().values
        for stat in stats:
            col_name = f'mo_p1_{feat}_{stat}'
            record[col_name] = _compute_agg(values, stat)

    # Meteo aggregation
    meteo_sample = meteo_p1[meteo_p1['sample_key'] == sk]
    for feat in meteo_feature_cols:
        if feat not in meteo_sample.columns:
            continue
        feat_type = get_feature_type(feat)
        stats = agg_cfg.get(feat_type, ['mean'])
        values = meteo_sample[feat].dropna().values
        for stat in stats:
            col_name = f'mo_p1_{feat}_{stat}'
            record[col_name] = _compute_agg(values, stat)

    all_sample_records.append(record)

infield_agg = pd.DataFrame(all_sample_records)
print(f'Aggregated features: {len(infield_agg)} samples x {len(infield_agg.columns) - 1} features')

# Check which model features are available
spectral_meteo_features = [f for f in selected_features if f.startswith('mo_p1_')]
available = [f for f in spectral_meteo_features if f in infield_agg.columns]
missing = [f for f in spectral_meteo_features if f not in infield_agg.columns]
print(f'\nModel spectral/meteo features: {len(spectral_meteo_features)}')
print(f'  Available: {len(available)}')
if missing:
    print(f'  Missing: {missing}')

## 7. Combine All Features

In [ ]:
# Start with aggregated spectral + meteo features
infield_features = infield_agg.copy()

# Add metadata
infield_features = infield_features.merge(
    infield_meta[['sample_key', 'field_key', 'protein_pct']],
    on='sample_key', how='left'
)

# Add elevation (elev_min per sample)
if 'elev_min' in infield_elev.columns:
    infield_features = infield_features.merge(
        infield_elev[['sample_key', 'elev_min']],
        on='sample_key', how='left'
    )
elif 'elev_mean' in infield_elev.columns:
    # Fallback: use elev_mean as elev_min proxy
    elev_temp = infield_elev[['sample_key']].copy()
    elev_temp['elev_min'] = infield_elev['elev_mean']
    infield_features = infield_features.merge(elev_temp, on='sample_key', how='left')
    print('WARNING: Using elev_mean as proxy for elev_min')

# Add soil features (matched via coordinates in previous cell)
infield_features = infield_features.merge(
    infield_soil_clean,
    on='sample_key', how='left'
)

# Check all 18 features
print('Feature availability check:')
for f in selected_features:
    if f in infield_features.columns:
        n_nan = infield_features[f].isna().sum()
        print(f'  {f:40s} OK   (NaN: {n_nan}/{len(infield_features)})')
    else:
        print(f'  {f:40s} MISSING')

print(f'
Final in-field features: {len(infield_features)} samples')

## 8. Train Model & Predict

Train RandomForest on **all** field-level training data (no CV holdout),
then predict on in-field samples.

In [ ]:
# Prepare training data
train_valid = train_df.dropna(subset=selected_features + ['protein_pct'])
X_train = train_valid[selected_features].values
y_train = train_valid['protein_pct'].values
print(f'Training: {len(X_train)} fields')

# Prepare in-field data
infield_valid = infield_features.dropna(subset=selected_features + ['protein_pct'])
X_infield = infield_valid[selected_features].values
y_infield = infield_valid['protein_pct'].values
print(f'In-field: {len(X_infield)} samples '
      f'(dropped {len(infield_features) - len(infield_valid)} with NaN)')

# Train model on all training data
rf = RandomForestRegressor(random_state=42, n_jobs=-1, **best_params)
rf.fit(X_train, y_train)
print(f'\nModel trained with {rf.n_estimators} trees')

# Predict on in-field samples
y_pred = rf.predict(X_infield)

# Metrics
r2 = r2_score(y_infield, y_pred)
rmse = np.sqrt(mean_squared_error(y_infield, y_pred))
mbe = np.mean(y_pred - y_infield)

print(f'\n{"="*50}')
print(f'IN-FIELD VALIDATION RESULTS')
print(f'{"="*50}')
print(f'R2:   {r2:.4f}')
print(f'RMSE: {rmse:.4f} %')
print(f'MBE:  {mbe:.4f} %')
print(f'{"="*50}')

# Store results for plotting
results_df = infield_valid[['sample_key', 'field_key', 'protein_pct']].copy()
results_df['predicted'] = y_pred
results_df['residual'] = y_pred - y_infield

## 9. Visualizations

In [ ]:
# === Scatter plot: predicted vs observed ================================
fig, ax = plt.subplots(figsize=(8, 8))

# Color by field
fields = sorted(results_df['field_key'].unique())
colors = plt.cm.tab20(np.linspace(0, 1, len(fields)))
field_color_map = dict(zip(fields, colors))

for fk in fields:
    mask = results_df['field_key'] == fk
    ax.scatter(
        results_df.loc[mask, 'protein_pct'],
        results_df.loc[mask, 'predicted'],
        c=[field_color_map[fk]],
        label=f'Field {fk}',
        s=30, alpha=0.7, edgecolors='white', linewidth=0.5
    )

# 1:1 line
lims = [
    min(ax.get_xlim()[0], ax.get_ylim()[0]),
    max(ax.get_xlim()[1], ax.get_ylim()[1])
]
ax.plot(lims, lims, 'k--', alpha=0.5, label='1:1 line')
ax.set_xlim(lims)
ax.set_ylim(lims)

ax.set_xlabel('Observed Protein (%)', fontsize=12)
ax.set_ylabel('Predicted Protein (%)', fontsize=12)
ax.set_title(
    f'In-Field Validation: RF mo_p1\n'
    f'R$^2$={r2:.3f}, RMSE={rmse:.3f}%, MBE={mbe:.3f}%',
    fontsize=13
)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8, ncol=2)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/14_infield_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === Per-field analysis ===================================================
field_stats = results_df.groupby('field_key').agg(
    n_samples=('protein_pct', 'count'),
    obs_mean=('protein_pct', 'mean'),
    obs_std=('protein_pct', 'std'),
    pred_mean=('predicted', 'mean'),
    pred_std=('predicted', 'std'),
    rmse=('residual', lambda x: np.sqrt(np.mean(x**2))),
    mbe=('residual', 'mean'),
).round(3)

# Per-field R2 (only for fields with >2 samples and variance)
field_r2 = []
for fk, grp in results_df.groupby('field_key'):
    if len(grp) > 2 and grp['protein_pct'].std() > 0:
        field_r2.append({
            'field_key': fk,
            'r2': r2_score(grp['protein_pct'], grp['predicted'])
        })
    else:
        field_r2.append({'field_key': fk, 'r2': np.nan})

field_r2_df = pd.DataFrame(field_r2).set_index('field_key')
field_stats = field_stats.join(field_r2_df)

print('Per-field statistics:')
print(field_stats.to_string())
print(f'\nMean field RMSE: {field_stats["rmse"].mean():.3f}')
print(f'Mean field R2: {field_stats["r2"].mean():.3f}')

In [ ]:
# === Boxplot: observed vs predicted per field ============================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Observed
results_df.boxplot(column='protein_pct', by='field_key', ax=axes[0])
axes[0].set_title('Observed Protein (%) by Field')
axes[0].set_xlabel('Field')
axes[0].set_ylabel('Protein (%)')
plt.sca(axes[0])
plt.xticks(rotation=45)

# Predicted
results_df.boxplot(column='predicted', by='field_key', ax=axes[1])
axes[1].set_title('Predicted Protein (%) by Field')
axes[1].set_xlabel('Field')
axes[1].set_ylabel('Protein (%)')
plt.sca(axes[1])
plt.xticks(rotation=45)

fig.suptitle('In-Field Protein Distribution by Field', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../figures/14_infield_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === Within-field variability: observed vs predicted per sample ==========
n_fields = len(results_df['field_key'].unique())
ncols = 4
nrows = (n_fields + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.5 * nrows))
axes = axes.flatten()

for idx, fk in enumerate(sorted(results_df['field_key'].unique())):
    if idx >= len(axes):
        break
    ax = axes[idx]
    fk_data = results_df[results_df['field_key'] == fk].sort_values('protein_pct')

    x = range(len(fk_data))
    ax.scatter(x, fk_data['protein_pct'], c='blue', s=20, label='Observed', zorder=3)
    ax.scatter(x, fk_data['predicted'], c='red', s=20, marker='^', label='Predicted', zorder=3)

    # Connect observed-predicted pairs
    for i, (_, row) in enumerate(fk_data.iterrows()):
        ax.plot([i, i], [row['protein_pct'], row['predicted']],
               'gray', alpha=0.3, linewidth=0.5)

    fk_r2 = field_stats.loc[fk, 'r2'] if fk in field_stats.index else np.nan
    r2_str = f'{fk_r2:.2f}' if np.isfinite(fk_r2) else 'N/A'
    ax.set_title(f'Field {fk} (n={len(fk_data)}, R2={r2_str})', fontsize=9)
    ax.tick_params(labelsize=7)
    if idx == 0:
        ax.legend(fontsize=7)

# Hide unused axes
for idx in range(n_fields, len(axes)):
    axes[idx].set_visible(False)

fig.suptitle('Within-Field: Observed vs Predicted Protein per Sample', fontsize=14)
plt.tight_layout()
plt.savefig('../figures/14_infield_within_field.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Feature Importance

In [ ]:
# Feature importance from the trained RF model
importances = pd.Series(rf.feature_importances_, index=selected_features)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
importances.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Feature Importance (MDI)')
ax.set_title('Random Forest Feature Importance')
plt.tight_layout()
plt.savefig('../figures/14_infield_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === Residual analysis ====================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Residual distribution
axes[0].hist(results_df['residual'], bins=30, edgecolor='black', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_xlabel('Residual (Predicted - Observed)')
axes[0].set_ylabel('Count')
axes[0].set_title('Residual Distribution')

# Residual vs observed
axes[1].scatter(results_df['protein_pct'], results_df['residual'], s=15, alpha=0.5)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Observed Protein (%)')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residual vs Observed')

# Residual by field
results_df.boxplot(column='residual', by='field_key', ax=axes[2])
axes[2].axhline(0, color='red', linestyle='--')
axes[2].set_title('Residuals by Field')
axes[2].set_xlabel('Field')
plt.sca(axes[2])
plt.xticks(rotation=45)

fig.suptitle('Residual Analysis', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../figures/14_infield_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === Field-level prediction: mean of sample predictions vs field mean ====
field_level = results_df.groupby('field_key').agg(
    obs_field_mean=('protein_pct', 'mean'),
    pred_field_mean=('predicted', 'mean'),
    n_samples=('sample_key', 'count'),
).reset_index()

r2_field = r2_score(field_level['obs_field_mean'], field_level['pred_field_mean'])
rmse_field = np.sqrt(mean_squared_error(field_level['obs_field_mean'], field_level['pred_field_mean']))

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(field_level['obs_field_mean'], field_level['pred_field_mean'],
          s=field_level['n_samples'] * 10, alpha=0.7, edgecolors='black')

for _, row in field_level.iterrows():
    ax.annotate(f"F{row['field_key']}",
               (row['obs_field_mean'], row['pred_field_mean']),
               fontsize=7, ha='left', va='bottom')

lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]),
        max(ax.get_xlim()[1], ax.get_ylim()[1])]
ax.plot(lims, lims, 'k--', alpha=0.5)
ax.set_xlim(lims)
ax.set_ylim(lims)

ax.set_xlabel('Observed Field Mean Protein (%)', fontsize=12)
ax.set_ylabel('Predicted Field Mean Protein (%)', fontsize=12)
ax.set_title(
    f'Field-Level Prediction (mean of in-field samples)\n'
    f'R$^2$={r2_field:.3f}, RMSE={rmse_field:.3f}%',
    fontsize=13
)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/14_infield_field_level_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Field-level R2: {r2_field:.4f}')
print(f'Field-level RMSE: {rmse_field:.4f}%')

## Summary

### Key Results
- **Model**: Random Forest, `mo_p1` temporal window, 18 features
- **Training**: Field-level CV R2 = 0.30 (228 fields)
- **In-field validation**: See metrics above

### Interpretation
- **Sample-level R2**: Can the model predict protein at individual sample locations?
- **Field-level R2**: Does averaging sample predictions recover field-level accuracy?
- **Within-field R2**: Can the model capture the relative protein ranking within a field?
- Per-field RMSE shows prediction accuracy varies by field